Email Spam Detection

Problem Statement:

The objective of this project is to build a classification model that can automatically detect whether a given email/message is spam or not spam (ham). With the increasing use of digital communication, spam messages pose a major issue by wasting time and potentially spreading malicious content. This model aims to classify messages accurately using natural language processing (NLP) and machine learning techniques.

Dataset:
SMS Spam Collection Dataset

In [58]:
# Import Libraries
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [59]:
df = pd.read_csv("spam.csv", encoding='latin-1')

In [60]:
# Remove unwanted columns
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

In [61]:
# Rename columns
df.columns = ['label', 'message']

In [62]:
# Convert labels
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

In [4]:
print(df.head())

     v1                                                 v2 Unnamed: 2  \
0   ham  Go until jurong point, crazy.. Available only ...        NaN   
1   ham                      Ok lar... Joking wif u oni...        NaN   
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...        NaN   
3   ham  U dun say so early hor... U c already then say...        NaN   
4   ham  Nah I don't think he goes to usf, he lives aro...        NaN   

  Unnamed: 3 Unnamed: 4  
0        NaN        NaN  
1        NaN        NaN  
2        NaN        NaN  
3        NaN        NaN  
4        NaN        NaN  


In [63]:
# Remove duplicates
df.drop_duplicates(inplace=True)

In [64]:
# 3. TEXT PREPROCESSING
ps = PorterStemmer()
def preprocess(text):
    text = text.lower()
    text = re.sub('[^a-zA-Z]', ' ', text)
    words = text.split()
    words = [ps.stem(word) for word in words if word not in stopwords.words('english')]
    return " ".join(words)
df['cleaned_message'] = df['message'].apply(preprocess)

In [65]:
# 4. TRAIN-TEST SPLIT
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['cleaned_message'], df['label'], test_size=0.2, random_state=42
)

In [66]:
# 5. TF-IDF (NO DATA LEAKAGE)
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train = tfidf.fit_transform(X_train_text)
X_test = tfidf.transform(X_test_text)

In [67]:
# 6. MODEL
base_model = LogisticRegression(max_iter=1000)
model = CalibratedClassifierCV(base_model)
model.fit(X_train, y_train)

CalibratedClassifierCV(estimator=LogisticRegression(max_iter=1000))

In [68]:
# 7. EVALUATION
print(classification_report(y_test, model.predict(X_test)))

              precision    recall  f1-score   support

           0       0.99      0.98      0.99       889
           1       0.88      0.94      0.91       145

    accuracy                           0.97      1034
   macro avg       0.94      0.96      0.95      1034
weighted avg       0.98      0.97      0.98      1034



In [69]:
# 8. PREDICTION FUNCTION
threshold = 0.6
def predict_spam(text):
    text = preprocess(text)
    vector = tfidf.transform([text])
    prob = model.predict_proba(vector)[0][1]
    # safety clipping
    prob = min(max(prob, 0.01), 0.99)
    if prob > threshold:
        return f"Spam ({prob:.2f})"
    else:
        return f"Not Spam ({prob:.2f})"

In [70]:
# 9. TEST CASES
print(predict_spam("Hey, are we meeting today?"))
print(predict_spam("URGENT! You won 5000 cash, call now"))
print(predict_spam("Free entry in a competition, claim prize"))

Not Spam (0.01)
Spam (0.99)
Spam (0.99)


In [71]:
import pickle
# Save model
pickle.dump(model, open("model.pkl", "wb"))
# Save TF-IDF
pickle.dump(tfidf, open("vectorizer.pkl", "wb"))